# Sandbox - Hyper-Parameters optimisation

This notebook is used to prototype the Hyper-Parameters optimisation process, in all scenarios of use of the algorithm.

---

## Imports & Config

In [1]:
! pwd

/Users/simonlejoly/Documents/Work/mimosa/tests


In [2]:
! export XLA_PYTHON_CLIENT_MEM_FRACTION=.25

In [3]:
# Jax configuration
USE_JIT = True
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [4]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from typing import Tuple

In [5]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)
import jax.random as jr
import jax.numpy as jnp
import jax.lax as lax
import jax.scipy as jsp
from jax import vmap, jit, Array, grad

from equinox import filter_jit
import numpy as np

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.dpi'] = 300
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kernax import WhiteNoiseKernel, VarianceKernel, SEKernel, AffineMean, AbstractMean, AbstractKernel
from kernax.mask import create_mask

In [6]:
%matplotlib inline

In [7]:
# Local imports
from mimosa.linalg import cho_factor, cho_solve
from mimosa.generate_data import generate_data
from mimosa.hyperpost import hyperpost
from mimosa.sampling import sample_gp
from mimosa.nll import means_nlls, tasks_nlls, full_nll

2026-04-22 21:38:49,467 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [8]:
# Config
key = jr.PRNGKey(42)

T=120 ; K=3 ; F=1 ; N=75 ; I=1 ; O=2 ; gs=250 if I == 1 else 40

sth=True ; sch=True ; chit=False ; fh=False ; soh=True ; siit=False ; siif=True

mean = AffineMean(slope=0., intercept=0.)
mean_kernel = VarianceKernel(20.) * SEKernel(length_scale=10.)
task_kernel = VarianceKernel(.2) * SEKernel(length_scale=9.) + WhiteNoiseKernel(noise=.01)

mean_priors = {
	"slope": (-.2, .2),
	"intercept": (-2.5, 2.5)
}

mean_kernel_priors = {
	"variance": (5, 10.),
	"length_scale": (2.5, 10.)
}

task_kernel_priors = {
	"variance": (0.25, 1.),
	"length_scale": (2., 8.),
	"noise": (0.01, 0.1)
}

jax.devices()

[CpuDevice(id=0)]

In [9]:
inputs, outputs, maps, grid, m_p_means, m_p_covs, m_p, mix, t_m, m, m_k, t_k = generate_data(
	key, T, K, F, N,  I, O, gs,
	mean, mean_kernel, task_kernel,
	mean_priors, mean_kernel_priors, task_kernel_priors,
	sth, sch, chit, fh, soh, siit, siif)

In [10]:
mix_coeffs = jnp.eye(K)[mix]
mix_coeffs.shape

(120, 3)

In [11]:
p_m, p_c = hyperpost(inputs, outputs, maps, grid, mix_coeffs, m, m_k, t_k)
p_m.shape, p_c.shape

/opt/miniconda3/envs/mimosa/lib/python3.13/site-packages/jax/_src/scipy/linalg.py:167: FutureWarning: jax.scipy.linalg.cho_solve: batched 1D solves with b.ndim > 1 are deprecated, and in the future will be treated as a batched 2D solve. Use cho_solve(c_and_lower, b[..., None]).squeeze(-1) to avoid this warning.
  warnings.warn(


((3, 2, 250), (3, 1, 250, 250))

In [12]:
print(t_k)

VarianceKernel(variance=0.98) * SEKernel(length_scale=2.01) + WhiteNoiseKernel(noise=0.03)


## Prediction

In [13]:
def predict_task_output(output_obs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid):
	""" Predicts a single task output, within a single cluster"""
	padding_mask_1D = ~jnp.isnan(output_obs)[:, None]
	padding_mask_2D = padding_mask_1D & padding_mask_1D.T

	gamma_obs = jnp.where(padding_mask_2D, gamma_obs, jnp.eye(len(gamma_obs)))
	gamma_crossed = jnp.where(padding_mask_1D, gamma_crossed, 0.)

	gamma_pred_U = cho_factor(gamma_obs)
	z = lax.linalg.triangular_solve(gamma_pred_U, gamma_crossed.T).T
	y = lax.linalg.triangular_solve(gamma_pred_U, jnp.nan_to_num(output_obs) - post_mean_obs)

	pred_mean = post_mean_grid + (z.T @ y)
	pred_cov = gamma_grid - (z.T @ z)

	return pred_mean, pred_cov

In [14]:
def predict_task_in_cluster(output_obs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid):
	if gamma_obs.shape[0] == 1:
		return (vmap(
			predict_task_output,
			in_axes=(0, 0, 0, None, None, None))
		        (output_obs.T, post_mean_obs, post_mean_grid, gamma_obs[0], gamma_crossed[0], gamma_grid[0]))
	else:
		return (vmap(
			predict_task_output,
			in_axes=(0, 0, 0, 0, 0, 0))
		        (output_obs.T, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid))

In [15]:
def predict_clusters(task_outputs, task_mappings,
                     post_mean_grid,
                     post_cov_grid,
                     task_cov_obs, task_cov_grid, task_cov_crossed):
	post_mean_obs = post_mean_grid[:, :, task_mappings]
	post_cov_obs = post_cov_grid[:, :, task_mappings, :][:, :, :, task_mappings]
	post_cov_crossed = post_cov_grid[:, :, task_mappings, :]

	gamma_obs = post_cov_obs + task_cov_obs
	gamma_grid = post_cov_grid + task_cov_grid
	gamma_crossed = post_cov_crossed + task_cov_crossed

	if gamma_obs.shape[0] == 1:
		return vmap(
			predict_task_in_cluster,
			in_axes=(None, 0, 0, None, None, None)
		)(task_outputs, post_mean_obs, post_mean_grid, gamma_obs[0], gamma_crossed[0], gamma_grid[0])
	return vmap(
		predict_task_in_cluster,
		in_axes=(None, 0, 0, 0, 0, 0)
	)(task_outputs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid)

In [16]:
def predict(outputs, mappings,
            post_mean_grid,
            post_cov_grid,
            tasks_cov_obs, tasks_cov_grid, tasks_cov_crossed):
	""" Predict every task for every cluster. """
	if tasks_cov_obs.shape[0] == 1:
		return vmap(
			predict_clusters,
			in_axes=(0, 0,
			         None,
			         None,
			         None, None, None),
		)(outputs, mappings,
		  post_mean_grid,
		  post_cov_grid,
		  tasks_cov_obs[0], tasks_cov_grid[0], tasks_cov_crossed[0])

	return vmap(
		predict_clusters,
		in_axes=(0, 0,
		         None,
		         None,
		         0, 0, 0),
	)(outputs, mappings,
	  post_mean_grid,
	  post_cov_grid,
	  tasks_cov_obs, tasks_cov_grid, tasks_cov_crossed)

In [17]:
t_k_noiseless = t_k.replace(noise=1e-12)
extended_grid = jnp.broadcast_to(grid, (len(inputs),)+grid.shape)

In [20]:
pred_means, pred_covs = predict(outputs, maps, p_m, p_c, t_k(inputs), t_k_noiseless(extended_grid), t_k_noiseless(inputs, extended_grid))
pred_means.shape, pred_covs.shape

((120, 3, 2, 250), (120, 3, 2, 250, 250))

In [ ]:
# Shape match : (Task, Cluster, Output, Grid, Grid)

In [ ]:
# CLAUDE: Place your code here


In [19]:
0/0

ZeroDivisionError: division by zero

First, try to predict a single task, within a single cluster, for a single output.

In [ ]:
t = 1 ; task_mix = jnp.array(0) ; o = 1

In [ ]:
task_inputs = inputs[t][:N//2, :]
task_outputs = outputs[t][:N//2, :]
task_map = maps[t][:N//2]
#task_mix = mix[t]

task_inputs_test = inputs[t][N//2:, :]
task_outputs_test = outputs[t][N//2:, :]
task_map_test = maps[t][N//2:]

task_inputs.shape, task_outputs.shape, task_map.shape, task_mix.shape

In [ ]:
m(grid).shape

In [ ]:
m_p.shape

In [ ]:
plt.plot(task_inputs, task_outputs[:, o], 'g.', label="Known Observations")
plt.plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
plt.plot(grid, m(grid)[task_mix, o, :], 'b--', alpha=0.3, label="Hyperprior Mean")
plt.plot(grid, m_p[task_mix, o, :], 'b-', alpha=0.5, label="Hyperposterior Mean")
plt.legend()
plt.title(f"Task {t}, output {o}")
plt.show()

In [ ]:
task_p_m = p_m[task_mix][:, task_map]
task_p_c = p_c[task_mix][:, task_map, :][:, :, task_map]

task_p_m.shape, task_p_c.shape

In [ ]:
p_m_obs = p_m[task_mix][:, task_map]
p_m_grid = p_m[task_mix]  # here, the grid is all_inputs, but it could be any set of inputs as long as the mappings are re-computed.
p_c_obs = p_c[task_mix][:, task_map, :][:, :, task_map]
p_c_grid = p_c[task_mix]
p_c_crossed = p_c[task_mix][:, task_map, :]

p_m_obs.shape, p_m_grid.shape, p_c_obs.shape, p_c_grid.shape, p_c_crossed.shape

In [ ]:
task_cov = t_k(inputs)[t, task_mix]  # Because specific task, cluster and output
task_cov_obs = task_cov[:, :N//2, :][:, :, :N//2]
task_cov.shape, task_cov_obs.shape

In [ ]:
t_k_noiseless = t_k.replace(noise=1e-12)

task_cov_grid = t_k_noiseless(jnp.broadcast_to(grid, (len(inputs),)+grid.shape))[t, task_mix]
task_cov_crossed = t_k_noiseless(inputs, jnp.broadcast_to(grid, (len(inputs),)+grid.shape))[t, task_mix]
task_cov_crossed = task_cov_crossed[:, :N//2, :]

task_cov_grid.shape, task_cov_crossed.shape

In [ ]:
t_k_noiseless = t_k.replace(noise=1e-12)

task_cov_grid = t_k_noiseless(jnp.broadcast_to(grid, (len(inputs),)+grid.shape))
task_cov_crossed = t_k_noiseless(inputs, jnp.broadcast_to(grid, (len(inputs),)+grid.shape))

task_cov_grid.shape, task_cov_crossed.shape

In [ ]:
gamma_obs = p_c_obs + task_cov_obs
gamma_grid = p_c_grid + task_cov_grid
gamma_crossed = p_c_crossed + task_cov_crossed

gamma_obs.shape, gamma_grid.shape, gamma_crossed.shape, task_outputs.mT.shape

In [ ]:
pred_mean, pred_cov = predict_task_in_cluster(task_outputs, p_m_obs, p_m_grid, gamma_obs, gamma_crossed, gamma_grid)

In [ ]:
task_outputs.shape, p_m_obs.shape, p_m_grid.shape, gamma_obs.shape, gamma_crossed.shape, gamma_grid.shape

In [ ]:
pred_mean.shape, pred_cov.shape

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), nrows=1, ncols=2)
for o in range(O):
	ax[o].plot(task_inputs, task_outputs[:, o], 'g.', label="Known Observations")
	ax[o].plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
	ax[o].plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
	ax[o].plot(grid, m(grid)[task_mix, o, :], 'b--', alpha=0.3, label="Hyperprior Mean")
	ax[o].plot(grid, m_p[task_mix, o, :], 'b-', alpha=0.5, label="Hyperposterior Mean")
	ax[o].plot(grid, pred_mean[o], 'g-', label="Prediction mean")
	ax[o].fill_between(
		grid.squeeze(),
		pred_mean[o] - 1.98 * jnp.diag(pred_cov[o]),
		pred_mean[o] + 1.98 * jnp.diag(pred_cov[o]),
		label="Prediction CI", color='g', alpha=0.2)
	ax[o].legend()
	ax[o].set_ylim(-15, 15)
fig.suptitle(f"Task {1}")
plt.show()

---

In [ ]:
0/0

In [ ]:
task_p_m = p_m[task_mix][o][task_map]
task_p_c = p_c[task_mix][o][task_map, :][:, task_map]

task_p_m.shape, task_p_c.shape

In [ ]:
p_m_obs = p_m[task_mix][o][task_map]
p_m_grid = p_m[task_mix][o]  # here, the grid is all_inputs, but it could be any set of inputs as long as the mappings are re-computed.
p_c_obs = p_c[task_mix][o][task_map, :][:, task_map]
p_c_grid = p_c[task_mix][o][:, :]
p_c_crossed = p_c[task_mix][o][task_map, :][:, :]

p_m_obs.shape, p_m_grid.shape, p_c_obs.shape, p_c_grid.shape, p_c_crossed.shape

In [ ]:
task_cov = t_k(inputs)[t, task_mix, o]  # Because specific task, cluster and output
task_cov_obs = task_cov[:N//2, :][:, :N//2]
task_cov.shape, task_cov_obs.shape

In [ ]:
t_k_noiseless = t_k.replace(noise=1e-12)

task_cov_grid = t_k_noiseless(jnp.broadcast_to(grid, (len(inputs),)+grid.shape))[t, task_mix, o]
task_cov_crossed = t_k_noiseless(inputs, jnp.broadcast_to(grid, (len(inputs),)+grid.shape))[t, task_mix, o]
task_cov_crossed = task_cov_crossed[:N//2, :]

task_cov_grid.shape, task_cov_crossed.shape

In [ ]:
gamma_obs = p_c_obs + task_cov_obs
gamma_grid = p_c_grid + task_cov_grid
gamma_crossed = p_c_crossed + task_cov_crossed

gamma_obs.shape, gamma_grid.shape, gamma_crossed.shape, task_outputs.mT.shape

In [ ]:
padding_mask = ~jnp.isnan(task_outputs[:, o])[:, None] & ~jnp.isnan(task_outputs[:, o])[None, :]
gamma_obs = jnp.where(padding_mask, gamma_obs, jnp.eye(len(gamma_obs)))
gamma_crossed = jnp.where(~jnp.isnan(task_outputs[:, o])[:, None], gamma_crossed, 0.)

In [ ]:
gamma_pred_U = cho_factor(gamma_obs)
z = lax.linalg.triangular_solve(gamma_pred_U, gamma_crossed.T).T
y = lax.linalg.triangular_solve(gamma_pred_U, jnp.nan_to_num(task_outputs[:, o]) - p_m_obs)

pred_mean = p_m_grid + (z.T @ y)
pred_cov = gamma_grid - (z.T @ z)

In [ ]:
pred_mean.shape, pred_cov.shape

In [ ]:
plt.plot(task_inputs, task_outputs[:, o], 'g.', label="Known Observations")
plt.plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
plt.plot(grid, m(grid)[task_mix, o], 'b--', alpha=0.3, label="Hyperprior Mean")
plt.plot(grid, m_p[task_mix, o], 'b-', alpha=0.5, label="Hyperposterior Mean")
plt.plot(grid, pred_mean, 'g-', label="Prediction mean")
plt.fill_between(grid.squeeze(), pred_mean - 1.98 * jnp.diag(pred_cov), pred_mean + 1.98 * jnp.diag(pred_cov), label="Prediction CI", color='g', alpha=0.2)
plt.legend()
plt.title(f"Task {1}, output {0}")
plt.show()

In [ ]:
o = 0

fig, ax = plt.subplots(figsize=(16, 6), nrows=1, ncols=2)
for o in range(O):
	ax[o].plot(task_inputs, task_outputs[:, o], 'g.', label="Known Observations")
	ax[o].plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
	ax[o].plot(task_inputs_test, task_outputs_test[:, o], 'r.', label="Test Observations")
	ax[o].plot(grid, m(grid)[task_mix, o, :], 'b--', alpha=0.3, label="Hyperprior Mean")
	ax[o].plot(grid, m_p[task_mix, o, :], 'b-', alpha=0.5, label="Hyperposterior Mean")
	ax[o].legend()
plt.legend()
plt.title(f"Task {1}")
plt.show()

---